Installing the Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== Step 0: Load and preprocess training data ====

# Define base paths
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)  # Resize to manage memory

train_data = []
train_labels = []

def process_session(base_path, session_label):
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Session {session_label}"):
        # 🔄 Updated to use image indices 01 to 05
        for img_idx in range(1, 6):
            fused_vector = []

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                folder_path = os.path.join(base_path, folder_name)
                img_filename = f"{img_idx:02d}.jpg"
                img_path = os.path.join(folder_path, img_filename)

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Missing image: {img_path}")
                    continue

                print(f"✅ Using image: {img_path}")

                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                fused_vector.append(img_norm.flatten())

            if len(fused_vector) == 4:
                final_vector = np.concatenate(fused_vector)
                train_data.append(final_vector)
                train_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")
            else:
                print(f"⚠️ Incomplete fusion: Subject {subject_id}, Image {img_idx}, Session {session_label}")

# Process both sessions using images 1–5
process_session(base_path_sess1, session_label=1)
process_session(base_path_sess2, session_label=2)

# Convert to arrays
train_data = np.array(train_data)
train_labels = np.array(train_labels)

print("✅ Training data shape:", train_data.shape)
print("✅ Training labels shape:", train_labels.shape)


Train:

In [ ]:

import numpy as np

# Step 1: Center the training data
mean_vector = np.mean(train_data, axis=0)
centered_data = train_data - mean_vector  # Shape: (n_samples, n_features)

# Step 2: Compute subject-to-subject Gram matrix
gram_matrix = centered_data @ centered_data.T  # Shape: (n_samples, n_samples)

# Step 3: Eigen decomposition of Gram matrix
eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)  # ascending order

# Step 4: Sort eigenvalues/vectors in descending order
sorted_indices = np.argsort(-eig_vals)
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

# Step 5: Filter valid eigenvectors (non-zero eigenvalues)
valid_indices = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid_indices]
eig_vecs_valid = eig_vecs[:, valid_indices]

# ✅ Step 6: Project all valid eigenvectors back to original feature space
eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

# Step 7: Project training data into full PCA space
train_data_pca = centered_data @ eig_vecs_full  # Shape: (n_samples, k)

# Final output
print("✅ PCA-transformed training data shape:", train_data_pca.shape)
print("✅ Number of principal components used:", eig_vecs_full.shape[1])


Preprocessing for Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIG (Ensure this matches training settings) ====
NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)

base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

# ==== RE-INITIALIZE STORAGE ====
test_data = []
test_labels = []
test_paths = []

# ==== LOOP: ONLY IMAGE 6 FROM BOTH SESSIONS ====
for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Preparing test data"):
    img_idx = 6  # ✅ Only image 6
    for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
        fused_vector = []
        current_paths = []

        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            img_path = os.path.join(base_path, folder_name, f"{img_idx:02d}.jpg")

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Missing: {img_path}")
                continue

            print(f"✅ Using: {img_path}")
            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            fused_vector.append(img_norm.flatten())
            current_paths.append(img_path)

        if len(fused_vector) == 4:
            sample_vector = np.concatenate(fused_vector)
            test_data.append(sample_vector)
            test_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")
            test_paths.append(current_paths[-1])
        else:
            print(f"⚠️ Incomplete: Subject {subject_id}, Image {img_idx}, Session {session_label}")

# ==== CONVERT TO NUMPY ARRAYS ====
test_data = np.array(test_data)
test_labels = np.array(test_labels)

print("\n✅ Test data loaded successfully.")
print(f"🧪 Total test samples: {len(test_data)}")
print(f"🧾 Example shape: {test_data[0].shape}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(test_data)  # 📊 Should be 123 if one test vector per subject

# 📉 Project test data into PCA space
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full

# 🔍 Loop through each test sample
for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "001_img5_s1"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # 🏆 Find closest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "001_img2_s1"

    # 🎯 Parse ID and session
    true_id, true_session = true_label.split('_')[0], true_label.split('_')[-1]
    pred_id, pred_session = predicted_label.split('_')[0], predicted_label.split('_')[-1]

    # ✅ Check match condition
    if pred_id == true_id and pred_session == true_session:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# 📈 Final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Recognition accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")


Benchmarking 2:

In [ ]:
correct_matches = 0
total_tests = len(test_data)  # e.g., 123 if one test vector per subject

# 📉 Project test data into PCA space
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full

# 🔍 Loop through each test sample
for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "047_img6_s1"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # 🏆 Find closest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "047_img2_s2"

    # 🎯 Extract only person IDs
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    # ✅ Person identification match (ignore session/image)
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# 📈 Final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
